# Nurikabe sampling trajectory

Load a Nurikabe LoRA adapter, fuse it into the base DiffusionGemma model, then
record a real denoising run and inspect the
**sampled string with 10 fields per row** and the **rendered grid** side by side.
Every intermediate state comes from the uniform categorical diffusion sampler.
The grid displays the sampled values, including mistakes, without validity
colours, clue repair, target overlays, or digit filtering.

The held-out example and preprocessing come from
[`configs/sft_nurikabe.py`](configs/sft_nurikabe.py), using its `eval_ds` pipeline.
This is the eval split used by that config; it is separate from training.
The base checkpoint, adapter, training step, example, exact prompt and sampling
settings are recorded in `manifest.json`.


## Settings

Assumes a GPU-enabled Python environment with the project and notebook
dependencies installed, plus a downloaded base DiffusionGemma checkpoint,
Nurikabe LoRA adapter, tokenizer and evaluation data. Set `BASE_CHECKPOINT` to
the base weights and `LORA_DIR` to the adapter bundle containing
`adapter_config.json` and its Orbax weights. The bundle is produced by
[`extract_nurikabe_lora.py`](../../../scripts/extract_nurikabe_lora.py) and can be
downloaded separately from Hugging Face.

Set `BAGZ_PATH` if the evaluation file is outside this checkout.
`GEMMA_CACHE_DIR` points to your existing tokenizer cache (default `~/.gemma`,
containing `tokenizer/tokenizer_gemma4.model`). Change `EXAMPLE_INDEX` or the
sampling settings below to adjust the trajectory.


In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
from IPython.display import HTML, FileLink, Video, display

PROJECT_DIR = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (path / "pyproject.toml").is_file()
                    and (path / "gemma" / "diffusion").is_dir()), None)
if PROJECT_DIR is None:
  raise FileNotFoundError("Run this notebook from inside the gemma checkout.")
sys.path.insert(0, str(PROJECT_DIR))
os.environ.setdefault("GEMMA_CACHE_DIR", str(Path.home() / ".gemma"))
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from gemma.diffusion.hackable_diffusion_adapter import puzzle_trajectory as pt
from gemma.diffusion.hackable_diffusion_adapter import lora_checkpoint as lc

TASK = "nurikabe"
# Base weights and adapter are downloaded separately.
BASE_CHECKPOINT = Path(os.environ.get(
    "DIFFUSIONGEMMA_CHECKPOINT", "/path/to/diffusiongemma-26B-A4B-it")).expanduser()
LORA_DIR = Path(os.environ.get(
    "PUZZLE_LORA_DIR", PROJECT_DIR / "nurikabe_lora")).expanduser()
BAGZ_PATH = Path(os.environ.get(
    "PUZZLE_BAGZ_PATH", PROJECT_DIR / "gemma" / "diffusion" /
    "hackable_diffusion_adapter" / "data" / TASK / f"{TASK}_eval.bagz")).expanduser()
EXAMPLE_INDEX = int(os.environ.get("PUZZLE_EXAMPLE_INDEX", "0"))
NUM_STEPS = int(os.environ.get("PUZZLE_NUM_STEPS", "64"))
SEED = 42
TEMPERATURE = 0.7
SECONDS_PER_STEP = float(os.environ.get("DIFFUSION_SECONDS_PER_STEP", "0.2"))
TRAJECTORY_DIR = Path(os.environ.get(
    "DIFFUSION_TRAJECTORY_DIR", PROJECT_DIR / "outputs" / f"{TASK}_trajectory"))
OUTPUT_PREFIX = os.environ.get("DIFFUSION_OUTPUT_PREFIX", f"{TASK}_denoising")
TRAJECTORY_DIR.mkdir(parents=True, exist_ok=True)
print(f"Trajectory and exports: {TRAJECTORY_DIR}")

## Load the example, base model and LoRA adapter

The config evaluation pipeline supplies the prompt tokens, including BOS, the
task-specific chat template and right padding. Only the dataset location,
a one-record slice, batch size 1 and workers 0 are changed.

Load the base weights and the adapter separately. The adapter metadata maps the
base checkpoint to the parameter names, shapes and dtypes used in training.
`peft.merge_params` combines the two trees; `lora.fuse_lora_params` adds each
LoRA update to its base weight. Sampling then uses the model without LoRA
wrappers. Only the small adapter bundle needs to be shared alongside this
notebook; the base model is obtained separately.


In [ ]:
import jax
import jax.numpy as jnp
from gemma import diffusion, gm, peft
from hackable_diffusion import hd
from kauldron import konfig
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_ar_state_handler
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_network
from gemma.diffusion.hackable_diffusion_adapter.hd import lora

if not any(device.platform == "gpu" for device in jax.devices()):
  raise RuntimeError("Sampling requires a GPU-enabled JAX environment.")
if not BASE_CHECKPOINT.is_dir():
  raise FileNotFoundError(f"Set BASE_CHECKPOINT to your local base weights: {BASE_CHECKPOINT}")

cfg, batch, example = pt.load_example(
    TASK, PROJECT_DIR, example_index=EXAMPLE_INDEX, bagz_path=BAGZ_PATH)
print(f"JAX {jax.__version__}; devices: {jax.devices()}")
print(f"Base checkpoint: {BASE_CHECKPOINT}")
print(f"Held-out data: {example['dataset']}; record {EXAMPLE_INDEX}")

adapter_info, adapter_params = lc.load_adapter(LORA_DIR)
if adapter_info["task"] != TASK:
  raise ValueError(f"Expected {TASK} adapter, found {adapter_info['task']}.")
base_params = gm.ckpts.load_params(BASE_CHECKPOINT, restore_concurrent_gb=16)
base_params = lc.align_base_params(base_params, adapter_info)
fused_params = lora.fuse_lora_params(peft.merge_params(base_params, adapter_params))
network_params = {"gemma_model": fused_params}
network = hd_gemma_network.WrappedDiffusionGemmaNetwork(
    gemma_model=diffusion.DiffusionGemma_26B_A4B())
jax.block_until_ready(network_params)
del base_params, adapter_params, fused_params
gc.collect()
tokenizer = gm.text.Gemma4Tokenizer()
print(f"Fused rank-{adapter_info['rank']} LoRA adapter from step {adapter_info['checkpoint_step']}.")


## Sample and save every denoising state

Use the config's uniform categorical corruption process, a linear schedule,
`DiscreteDDIMStep` with temperature 0.7, and the same self-conditioning as
`sample_ar_steps64` in `eval/ar_eval.py`. There is one 256-token canvas.
`NUM_STEPS=64` means 64 time points: the random start and 63 denoising updates.

`store_trajectory=True` retains logits as well as tokens temporarily, so this
uses more GPU memory than ordinary evaluation. Reduce `NUM_STEPS` if needed.
Only token IDs and decoded strings are saved to disk.

The displayed span starts at canvas position zero and ends at the **final**
canvas's first stop token, fixed across all states. No text is stripped from
inside that span. `tokens.npy` and `canvas_texts.json` retain the entire canvas,
including padding. `texts.json` retains each exact decoded response, so CPU
rendering does not need a tokenizer or model.

In [ ]:
if NUM_STEPS < 2:
  raise ValueError("NUM_STEPS must be at least 2.")
if cfg.aux.num_canvases != 1:
  raise ValueError("This notebook records one canvas; use a one-canvas config.")
CANVAS_LENGTH = int(cfg.aux.canvas_size)
PAD = tokenizer.special_tokens.PAD
stop_tokens = (tokenizer.special_tokens.EOS, tokenizer.special_tokens.END_OF_TURN,
               tokenizer.special_tokens.BEGIN_OF_TOOL_RESPONSE)
process = konfig.resolve(cfg.aux.corruption_process)
canvas_sampler = hd.sampling.DiffusionSampler(
    time_schedule=hd.sampling.UniformTimeSchedule(),
    stepper=hd.sampling.DiscreteDDIMStep(
        corruption_process=process, temperature=TEMPERATURE, logits_dtype=jnp.bfloat16),
    num_steps=NUM_STEPS, store_trajectory=True,
    update_conditioning_fn=hd_gemma_ar_state_handler.PropagateSelfConditioningFn())

inference_fn = hd.inference.FlaxLinenInferenceFn(network=network, params=network_params)

state_handler = hd_gemma_ar_state_handler.GemmaARStateHandler(
    gemma_network=network, gemma_params=network_params["gemma_model"],
    end_tokens=stop_tokens, pad_token=PAD)

prompt_tokens = jnp.asarray(batch["prompt"], dtype=jnp.int32)
prompt_lengths = jnp.sum(prompt_tokens != PAD, axis=1, dtype=jnp.int32)
started = time.monotonic()
sampler_state = state_handler.init_ar_state(
    batch_size=1, conditioning={"prompt_tokens": prompt_tokens, "prompt_lengths": prompt_lengths},
    canvas_length=CANVAS_LENGTH, max_num_canvases=1)
noise_rng, sample_rng = jax.random.split(jax.random.PRNGKey(SEED))
initial_noise = process.sample_from_invariant(
    key=noise_rng, data_spec=jnp.zeros((1, CANVAS_LENGTH, 1), dtype=jnp.int32))
print(f"Sampling {NUM_STEPS - 1} denoising updates...", flush=True)
last_step, trajectory = canvas_sampler(
    inference_fn=inference_fn, rng=sample_rng, initial_noise=initial_noise,
    conditioning=state_handler.create_conditioning_from_state(sampler_state=sampler_state))
trajectory_tokens = np.asarray(trajectory.xt[:, 0, :, 0], dtype=np.int32)
sampling_seconds = time.monotonic() - started
del last_step, trajectory
gc.collect()

np.save(TRAJECTORY_DIR / "prompt_tokens.npy", np.asarray(prompt_tokens))
manifest = pt.save_trajectory(
    TRAJECTORY_DIR, trajectory_tokens, tokenizer,
    dict(example, task=TASK, model="DiffusionGemma 26B-A4B (fused Nurikabe LoRA)",
         checkpoint=str(LORA_DIR / adapter_info["weights"]),
         checkpoint_step=int(adapter_info["checkpoint_step"]),
         base_checkpoint=str(BASE_CHECKPOINT), adapter=str(LORA_DIR),
         adapter_rank=int(adapter_info["rank"]),
         prompt=tokenizer.decode(np.asarray(prompt_tokens[0, :int(prompt_lengths[0])]).tolist()),
         prompt_len=int(cfg.aux.prompt_len), seed=SEED, num_steps=NUM_STEPS,
         temperature=TEMPERATURE, sampling_seconds=round(sampling_seconds, 2),
         corruption="uniform categorical, linear schedule",
         sampler="DiscreteDDIMStep, uniform time schedule, self-conditioning",
         jax=jax.__version__, device=str(jax.devices()[0])),
    stop_tokens=stop_tokens)
print(f"Saved {len(trajectory_tokens)} real canvas states in {sampling_seconds:.1f}s.")
print("Final response, with grid row breaks:\n" + pt.format_raw_grid(manifest["final_text"], pt.TASK_SIZES[TASK]))

## Inspect the saved trajectory (CPU)

All remaining cells read the saved run. To inspect or export it on CPU without
loading weights, run the settings cell and the cells below, with
`TRAJECTORY_DIR` pointing to the saved trajectory.

The raw view inserts a line break after every 10 whitespace-separated fields;
its expandable **Exact decoded response** preserves the original whitespace.
Unexpected words, invalid numbers and extra fields are retained. The grid places
the first 100 fields in row-major order, leaves missing cells empty, and
keeps extra fields in the string view. Nurikabe `#` is dark sea; every other
field is literal text. These views do not run a puzzle verifier or replace a
changed clue with its original value.


In [ ]:
run = pt.PuzzleTrajectory(TRAJECTORY_DIR)
if run.task != TASK:
  raise ValueError(f"Expected {TASK} trajectory, found {run.task}.")
print(f"Checkpoint: {run.info['checkpoint']}")
print(f"Held-out record: {run.info['example_index']} ({run.info['split']} split)")
print(f"States: {len(run.texts)}; updates: {run.updates}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, key, label in zip(axes, ("puzzle", "target"), ("Input puzzle", "Reference solution")):
  ax.imshow(pt.render_grid(TASK, run.info[key], side=540))
  ax.set_title(label)
  ax.axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Play/pause or scrub through every sampled state. The HTML is also portable.
interactive_path = TRAJECTORY_DIR / f"{OUTPUT_PREFIX}.html"
interactive = run.interactive_html(seconds_per_step=SECONDS_PER_STEP)
interactive_path.write_text("<!doctype html><meta charset='utf-8'>" + interactive)
display(HTML(interactive))
# If your notebook frontend blocks inline scripts, open the HTML in a browser.
display(FileLink(str(interactive_path)))

## Snapshot figure

Show the random start, one third, two thirds and final state. Each panel contains
both the row-formatted string and its grid. Save PNG/PDF copies for slides.

In [ ]:
snapshot_states = [round(fraction * run.updates) for fraction in (0, 1/3, 2/3, 1)]
fig, axes = plt.subplots(2, 2, figsize=(20, 11.25))
for ax, state in zip(axes.flat, snapshot_states):
  ax.imshow(run.render(state, size=(960, 540)))
  ax.axis("off")
fig.tight_layout(pad=0.3)
snapshot_path = TRAJECTORY_DIR / f"{OUTPUT_PREFIX}_trajectory.png"
fig.savefig(snapshot_path, dpi=160, facecolor="white")
fig.savefig(snapshot_path.with_suffix(".pdf"), facecolor="white")
plt.show()

plt.close(fig)
print(f"Saved {snapshot_path}")

## Export playback

Export a **1920×1080 H.264 MP4 at 30 fps**, a **960×540 looping GIF**, a final
poster PNG and a portable interactive HTML file. Both string and grid are visible
in every frame, with the true sampling step and field count. Playback holds the
start for one second and the end for three seconds; it is slower than sampling.
`export.json` records the timing and output names. Change `SECONDS_PER_STEP`
and rerun this cell to change the playback speed without sampling again.

In [ ]:
exports = run.export(prefix=OUTPUT_PREFIX, seconds_per_step=SECONDS_PER_STEP)
for kind, path in exports.items():
  print(f"{kind}: {path}")
display(Video(str(exports["mp4"]), embed=True, width=960))
display(FileLink(str(exports["gif"])))
display(FileLink(str(exports["html"])))